# Natural Language Processing
**Natural Language Processing or NLP** is a branch of Artificial Intelligence which deal with bridging the machines understanding humans in their Natural Language. Natural Language can be in form of text or sound, which are used for humans to communicate each other. NLP can enable humans to communicate to machines in a natural way. 


**Text Classification** is a process involved in Sentiment Analysis. It is classification of peoples opinion or expressions into different sentiments. Sentiments include *Positive, Neutral*, and *Negative*, *Review Ratings* and *Happy, Sad*. Sentiment Analysis can be done on different consumer centered industries to analyse people's opinion on a particular product or subject. 
![Sentiment Analysis](https://media-exp1.licdn.com/dms/image/C4D12AQHPAZFZZxBtng/article-cover_image-shrink_600_2000/0?e=1593648000&v=beta&t=eQAR5WOihE2_ZCCAJbsgNyJlaI_GW7u8lDw45zGbfuU)
> Sentiment Classification is a perfect problem in NLP for getting started in it. You can really learn a lot of concepts and techniques to master through doing project. Kaggle is a great place to learn and contribute your own ideas and creations. I learnt lot of things from other, now it's my turn to make document my project.

I will go through all the key and fundament concepts of NLP and Sequence Models, which you will learn in this notebook. 
![Sentiment Analysis](https://fiverr-res.cloudinary.com/images/t_main1,q_auto,f_auto,q_auto,f_auto/gigs/121192228/original/677c209a0a064cb9253973d3663684acf91dab84/do-nlp-projects-with-python-nltk-gensim.jpg)
Let's get started with code without furthur ado.


##  Importing Dependencies
   We shall start by importing all the neccessary libraries. I will explain the exact use of each library later in this notebook.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


import nltk 
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer



from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


import re

print("Tensorflow Version",tf.__version__)

#  Dataset Preprocessing
In this notebook, I am using **Sentiment-140** from [Kaggle](https://www.kaggle.com/kazanova/sentiment140). It contains a labels data of 1.6 Million Tweets and I find it a good amount of data to train our model.

In [ ]:
df = pd.read_csv('../input/sentiment140/training.1600000.processed.noemoticon.csv',
                 encoding = 'latin',header=None)
df.head()

You can see the columns are without any proper names. Lets rename them for our reference

In [ ]:
df.columns = ['sentiment', 'id', 'date', 'query', 'user_id', 'text']
df.head()

We are going to train only on text to classify its sentiment. So we can ditch the rest of the useless columns.

In [ ]:
df = df.drop(['id', 'date', 'query', 'user_id'], axis=1)

In [ ]:
lab_to_sentiment = {0:"Negative", 4:"Positive"}
def label_decoder(label):
  return lab_to_sentiment[label]
df.sentiment = df.sentiment.apply(lambda x: label_decoder(x))
df.head()

Here are decoding the labels. We map **0 -> Negative and 1 -> Positive** as directed by the datset desciption. Now that we decoded we shall now analyse the dataset by its distribution. Because it's important that we have almost small amount of examples for given classes.

In [ ]:
val_count = df.sentiment.value_counts()

plt.figure(figsize=(8,4))
plt.bar(val_count.index, val_count.values)
plt.title("Sentiment Data Distribution")

It's a very good dataset without any skewness. Thank Goodness.

Now let us explore the data we having here... 

In [ ]:
import random
random_idx_list = [random.randint(1,len(df.text)) for i in range(10)] # creates random indexes to choose from dataframe
df.loc[random_idx_list,:].head(10) # Returns the rows with the index and display it

Looks like we have a nasty data in text. Because in general we use lot of punctuations and other words without any contextual meaning. It have no value as feature to the model we are training. So we need to get rid of them.

# Text Preprocessing
Tweet texts often consists of other user mentions, hyperlink texts, emoticons and punctuations. In order to use them for learning using a Language Model. We cannot permit those texts for training a model. So we have to clean the text data using various preprocessing and cleansing methods. Let's continue
![Data Science Meme](https://miro.medium.com/max/800/1*Xhm9c9qDfXa3ZCQjiOvm_w.jpeg)


### Stemming/ Lematization
For grammatical reasons, documents are going to use different forms of a word, such as *write, writing and writes.* Additionally, there are families of derivationally related words with similar meanings. The goal of both stemming and lemmatization is to reduce inflectional forms and sometimes derivationally related forms of a word to a common base form.

Stemming usually refers to a process that chops off the ends of words in the hope of achieving goal correctly most of the time and often includes the removal of derivational affixes. 

Lemmatization usually refers to doing things properly with the use of a vocabulary and morphological analysis of words, normally aiming to remove inflectional endings only and to return the base and dictionary form of a word
![Stemming and Lematization](https://qph.fs.quoracdn.net/main-qimg-cd7f4bafaa42639deb999b1580bea69f)

### Hyperlinks and Mentions
Twitter is a social media platform where people can tag and mentions other people's ID and share videos and blogs from internet. So the tweets often contain lots of Hyperlinks and twitter mentions.

- Twitter User Mentions - Eg. @arunrk7, @andrewng
- Hyperlinks - Eg. https://keras.io, https://tensorflow.org

### Stopwords
Stopwords are commonly used words in English which have no contextual meaning in an sentence. So therefore we remove them before classification. Some stopwords are...
![Stopwords English](https://4.bp.blogspot.com/-yiEr-jCVv38/Wmk10d84DYI/AAAAAAAAk0o/IfgjfjpgrxM5NosUQrGw7PtLvgr6DAG8ACLcBGAs/s1600/Screen%2BShot%2B2018-01-24%2Bat%2B5.41.21%2BPM.png)

That looks like a tedious process, isn't?. Don't worry there is always some library in Python to do almost any work. The world is great!!!

**NLTK** is a python library which got functions to perform text processing task for NLP.



In [ ]:
from nltk.corpus import stopwords  # Importer la liste des mots vides en anglais depuis NLTK
from nltk.stem import SnowballStemmer  # Importer le Stemmer Snowball en anglais depuis NLTK
import re  # Importer la bibliothèque d'expressions régulières

# Définir la liste des mots vides en anglais en utilisant la bibliothèque NLTK
stop_words = stopwords.words('english')

# Créer une instance du Stemmer Snowball en anglais pour la racinisation
stemmer = SnowballStemmer('english')

# Définir une expression régulière pour nettoyer le texte, en supprimant les mentions, les liens et les caractères non alphanumériques
text_cleaning_re = "@\S+|https?:\S+|http?:\S|[^A-Za-z0-9]+"

In [ ]:
import re  # Importer la bibliothèque d'expressions régulières
from nltk.stem import SnowballStemmer  # Importer le Stemmer Snowball de la bibliothèque NLTK

# Définir une fonction appelée "preprocess" qui prend un texte en entrée et un paramètre optionnel "stem"
def preprocess(text, stem=False):
    # Nettoyer le texte en utilisant une expression régulière pour remplacer certains motifs par des espaces,
    # convertir le texte en minuscules et supprimer les espaces en début et en fin.
    text = re.sub(text_cleaning_re, ' ', str(text).lower()).strip()
    
    tokens = []  # Créer une liste vide appelée "tokens" pour stocker les mots traités
    
    # Diviser le texte nettoyé en mots et itérer sur chaque mot
    for token in text.split():
        if token not in stop_words:  # Vérifier si le mot n'est pas dans la liste des mots vides
            if stem:  # Si le paramètre "stem" est True
                tokens.append(stemmer.stem(token))  # Appliquer la racinisation au mot et l'ajouter à la liste "tokens"
            else:
                tokens.append(token)  # Sinon, ajouter le mot d'origine à la liste "tokens"
    
    # Joindre les mots traités dans la liste "tokens" en une seule chaîne de caractères avec des espaces entre eux
    return " ".join(tokens)  # Renvoyer le texte prétraité sous forme d'une seule chaîne de caractères


In [ ]:
# Appliquer la fonction "preprocess" à chaque élément de la colonne "text" du DataFrame (df)

# La colonne "text" du DataFrame (df) contient les données textuelles à prétraiter
# Nous utilisons la méthode "apply" pour appliquer une fonction à chaque élément de la colonne

df.text = df.text.apply(lambda x: preprocess(x))

# La fonction lambda prend un élément "x" de la colonne "text" comme entrée
# Elle applique la fonction "preprocess(x)" à cet élément pour nettoyer et prétraiter le texte

# Le résultat de cette opération est ensuite stocké à nouveau dans la colonne "text" du DataFrame,
# remplaçant ainsi les valeurs d'origine par les valeurs prétraitées


**Aaww.. It is clean and tidy now. Now let's see some word cloud visualizations of it.**

### Positive Words

In [ ]:
from wordcloud import WordCloud  # Importer la bibliothèque WordCloud pour créer un nuage de mots
import matplotlib.pyplot as plt  # Importer la bibliothèque Matplotlib pour l'affichage graphique

# Définir la taille de la figure pour l'affichage du nuage de mots
plt.figure(figsize=(20, 20))

# Créer un objet WordCloud avec certaines configurations
wc = WordCloud(max_words=2000, width=1600, height=800).generate(" ".join(df[df.sentiment == 'Positive'].text))

# Générer le nuage de mots en utilisant le texte des éléments où la colonne 'sentiment' est égale à 'Positive'
# La fonction ".join()" combine tous les textes en un seul texte

# Afficher le nuage de mots en utilisant Matplotlib
plt.imshow(wc, interpolation='bilinear')

# L'image du nuage de mots est affichée avec interpolation 'bilinear'

### Negative Words

In [ ]:
plt.figure(figsize = (20,20)) 
wc = WordCloud(max_words = 2000 , width = 1600 , height = 800).generate(" ".join(df[df.sentiment == 'Negative'].text))
plt.imshow(wc , interpolation = 'bilinear')

## Train and Test Split

In [ ]:
TRAIN_SIZE = 0.8  # TRAIN_SIZE est défini comme la proportion des données d'entraînement par rapport à l'ensemble total. Ici, il est défini à 0.8, ce qui signifie que 80 % des données seront utilisées pour l'entraînement.

MAX_NB_WORDS = 100000  # MAX_NB_WORDS représente le nombre maximal de mots à prendre en compte lors de la tokenisation des textes. Dans ce cas, il est fixé à 100 000, ce qui signifie que seuls les 100 000 mots les plus fréquents seront pris en compte.

MAX_SEQUENCE_LENGTH = 30  # MAX_SEQUENCE_LENGTH détermine la longueur maximale autorisée pour une séquence de mots. Les séquences plus longues seront tronquées ou remplies. Ici, la longueur maximale est de 30 mots.


In [ ]:
from sklearn.model_selection import train_test_split  # Importer la fonction pour diviser le jeu de données en ensembles d'entraînement et de test

# Diviser le DataFrame "df" en ensembles d'entraînement et de test en utilisant la fonction "train_test_split"
# La taille de l'ensemble de test est déterminée en soustrayant TRAIN_SIZE de 1 (complément à 1).

train_data, test_data = train_test_split(df, test_size=1-TRAIN_SIZE, random_state=7)

# La variable "train_data" contient l'ensemble d'entraînement, et "test_data" contient l'ensemble de test.

print("Train Data size:", len(train_data))  # Afficher la taille de l'ensemble d'entraînement (nombre d'échantillons)
print("Test Data size", len(test_data))  # Afficher la taille de l'ensemble de test (nombre d'échantillons)


`train_test_split` will shuffle the dataset and split it to gives training and testing dataset. It's important to shuffle our dataset before training.

In [ ]:
train_data.head(10)

# Tokenization
Given a character sequence and a defined document unit, tokenization is the task of chopping it up into pieces, called *tokens* , perhaps at the same time throwing away certain characters, such as punctuation. The process is called **Tokenization.**
![Tokenization](https://cdn.analyticsvidhya.com/wp-content/uploads/2019/11/tokenization.png)

`tokenizer` create tokens for every word in the data corpus and map them to a index using dictionary.

`word_index` contains the index for each word

`vocab_size` represents the total number of word in the data corpus

In [ ]:
from keras.preprocessing.text import Tokenizer  # Importer la classe Tokenizer de Keras

# Créer un objet Tokenizer
tokenizer = Tokenizer()

# Adapter le Tokenizer aux textes de l'ensemble d'entraînement
tokenizer.fit_on_texts(train_data.text)

# Récupérer l'index des mots à partir du Tokenizer
word_index = tokenizer.word_index

# Calculer la taille du vocabulaire en ajoutant 1 au nombre d'index de mots (pour prendre en compte l'index 0)
vocab_size = len(tokenizer.word_index) + 1

# Afficher la taille du vocabulaire
print("Taille du vocabulaire :", vocab_size)

Now we got a `tokenizer` object, which can be used to covert any word into a Key in dictionary (number).

Since we are going to build a sequence model. We should feed in a sequence of numbers to it. And also we should ensure there is no variance in input shapes of sequences. It all should be of same lenght. But texts in tweets have different count of words in it. To avoid this, we seek a little help from `pad_sequence` to do our job. It will make all the sequence in one constant length `MAX_SEQUENCE_LENGTH`.

In [ ]:
from keras.preprocessing.sequence import pad_sequences  # Importer la fonction de rembourrage des séquences de Keras

# Convertir les textes de l'ensemble d'entraînement en séquences de nombres en utilisant le tokenizer, puis les rembourrer à une longueur maximale définie
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data.text), maxlen=MAX_SEQUENCE_LENGTH)

# Convertir les textes de l'ensemble de test en séquences de nombres en utilisant le tokenizer, puis les rembourrer à une longueur maximale définie
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data.text), maxlen=MAX_SEQUENCE_LENGTH)

# Afficher la forme (dimensions) des données d'entraînement et de test
print("Forme de X d'entraînement :", x_train.shape)
print("Forme de X de test :", x_test.shape)


Supposons que vous ayez un ensemble de phrases en tant que données textuelles et que vous souhaitiez les convertir en séquences de nombres de longueur maximale 5, en utilisant un tokenizer et la fonction de rembourrage.

#### Ensemble de données d'entraînement :
1. "C'est un exemple de phrase."
2. "Une autre phrase simple."

#### Ensemble de test :
1. "Ceci est un test."
2. "Phrase pour tester."

Vous avez également défini `MAX_SEQUENCE_LENGTH` à 5.

##### Étape 1: Conversion en séquences de nombres (avec rembourrage pour atteindre une longueur de 5)

Pour l'ensemble d'entraînement, le tokenizer convertira les phrases en séquences de nombres, par exemple :
- "C'est un exemple de phrase." devient `[1, 2, 3, 4, 5]`
- "Une autre phrase simple." devient `[6, 7, 4, 8, 0] (rembourrage avec des zéros pour atteindre une longueur de 5)

##### Étape 2: Conversion des données de test (avec rembourrage)

Pour l'ensemble de test, le tokenizer convertira les phrases en séquences de nombres, par exemple :
- "Ceci est un test." devient `[9, 1, 3, 10, 0] (rembourrage avec des zéros pour atteindre une longueur de 5)
- "Phrase pour tester." devient `[5, 11, 12, 0, 0] (rembourrage avec des zéros pour atteindre une longueur de 5)

Une fois que vous avez effectué ces étapes, vous obtenez des matrices de données d'entraînement (`x_train`) et de test (`x_test`) où chaque ligne représente une séquence de nombres de longueur 5. Vous pouvez ensuite utiliser ces données pour l'entraînement de modèles de traitement de texte, par exemple, des modèles de réseaux neuronaux.


In [ ]:
# Extraire les valeurs uniques de la colonne "sentiment" de l'ensemble d'entraînement
labels = train_data.sentiment.unique().tolist()

# La variable "labels" contient maintenant la liste des différentes étiquettes de sentiment présentes dans l'ensemble d'entraînement.

### Label Encoding 
We are building the model to predict class in enocoded form (0 or 1 as this is a binary classification). We should encode our training labels to encodings.

In [ ]:
from sklearn.preprocessing import LabelEncoder  # Importer la classe LabelEncoder de scikit-learn

# Créer un objet LabelEncoder
encoder = LabelEncoder()

# Adapter l'encodeur aux étiquettes de sentiment de l'ensemble d'entraînement et de test
encoder.fit(train_data.sentiment.to_list())

# Transformer les étiquettes de sentiment de l'ensemble d'entraînement et de test en valeurs numériques
y_train = encoder.transform(train_data.sentiment.to_list())
y_test = encoder.transform(test_data.sentiment.to_list())

# Remodeler les tableaux pour avoir une dimension supplémentaire (-1 signifie que la taille est déterminée automatiquement)
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

# Afficher les formes (dimensions) des tableaux y_train et y_test
print("Forme de y_train :", y_train.shape)
print("Forme de y_test :", y_test.shape)

# Word Emdedding
In Language Model, words are represented in a way to intend more meaning and for learning the patterns and contextual meaning behind it. 

**Word Embedding** is one of the popular representation of document vocabulary.It is capable of capturing context of a word in a document, semantic and syntactic similarity, relation with other words, etc.

Basically, it's a feature vector representation of words which are used for other natural language processing applications.

We could train the embedding ourselves but that would take a while to train and it wouldn't be effective. So going in the path of Computer Vision, here we use **Transfer Learning**. We download the pre-trained embedding and use it in our model.

The pretrained Word Embedding like **GloVe & Word2Vec** gives more insights for a word which can be used for classification. If you want to learn more about the Word Embedding, please refer some links that I left at the end of this notebook.


In this notebook, I use **GloVe Embedding from Stanford AI** which can be found [here](https://nlp.stanford.edu/projects/glove/)

Le word embedding est une technique qui permet de représenter des mots sous forme de vecteurs numériques dans un espace continu. Ces vecteurs capturent la sémantique et les relations entre les mots.

#### Exemple de Word Embedding Simple :

Supposons que nous ayons un petit corpus de phrases :

1. "J'adore les chiens."
2. "Les chats sont mignons."
3. "Les chiens et les chats sont des animaux."

Nous souhaitons créer un word embedding pour quelques mots de ce corpus : "chats", "chiens", "animaux", "mignons".

Nous allons utiliser un exemple simplifié de word embedding en utilisant des vecteurs de dimension 2 pour représenter ces mots. Les vecteurs pourraient ressembler à ceci :

- "chats" : [1.2, 0.5]
- "chiens" : [0.9, 0.3]
- "animaux" : [0.3, 1.1]
- "mignons" : [1.0, 0.7]

Dans cet exemple, chaque mot est représenté par un vecteur 2D, où chaque dimension capture une caractéristique sémantique. Par exemple, la première dimension pourrait capturer la notion de "mignons" versus "non mignons", et la deuxième dimension pourrait capturer la notion de "animaux de compagnie" versus "animaux sauvages".

Maintenant, avec ces vecteurs de mots, nous pouvons effectuer des opérations sémantiques simples :

- "chiens" - "chats" donnerait un vecteur [0.3, -0.2], ce qui pourrait signifier la différence entre les chiens et les chats.
- "chats" + "mignons" donnerait un vecteur [2.2, 1.2], qui pourrait représenter quelque chose de "très mignon".

Ces exemples illustrent comment les vecteurs de mots capturent les relations sémantiques et permettent d'effectuer des opérations de manière sém


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
GLOVE_EMB = '/kaggle/working/glove.6B.300d.txt'  # Chemin du fichier GloVe contenant des vecteurs de mots pré-entraînés de dimension 300

EMBEDDING_DIM = 300  # Dimension des vecteurs de mots GloVe, qui est de 300 dans ce cas

LR = 1e-3  # Taux d'apprentissage (Learning Rate) défini à 0.001

BATCH_SIZE = 1024  # Taille du lot (batch size) utilisée lors de l'apprentissage

EPOCHS = 10  # Nombre d'époques pour l'entraînement du modèle

MODEL_PATH = '.../output/kaggle/working/best_model.hdf5'  # Chemin du modèle sauvegardé après l'entraînement

In [ ]:
embeddings_index = {}  # Créer un dictionnaire pour stocker les vecteurs de mots GloVe

f = open(GLOVE_EMB)  # Ouvrir le fichier GloVe contenant les vecteurs de mots

# Parcourir chaque ligne du fichier GloVe
for line in f:
    values = line.split()  # Diviser la ligne en mots et vecteurs
    word = values[0]  # Le premier élément de la ligne est le mot
    coefs = np.asarray(values[1:], dtype='float32')  # Les éléments restants sont les coefficients de vecteur convertis en tableau Numpy
    embeddings_index[word] = coefs  # Stocker le vecteur GloVe dans le dictionnaire sous la clé du mot

f.close()  # Fermer le fichier après lecture

# Afficher le nombre de vecteurs de mots GloVe chargés
print('Nombre de vecteurs de mots trouvés : %s' % len(embeddings_index))

In [ ]:
# Créer une matrice d'incorporation (embedding matrix) initialisée avec des zéros
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

# Parcourir le dictionnaire des mots (word_index) qui associe chaque mot à un index
for word, i in word_index.items():
    # Récupérer le vecteur de mot pré-entraîné à partir de embeddings_index
    embedding_vector = embeddings_index.get(word)
    
    # Si le vecteur de mot pré-entraîné existe, le copier dans la matrice d'incorporation
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

# À la fin de cette boucle, embedding_matrix contiendra les vecteurs GloVe pour les mots du vocabulaire.

#### Exemple d'Incorporation de Mots (Word Embedding) avec Matrice de Sortie

Supposons que nous ayons un vocabulaire de mots composé de trois mots : "chat", "chien" et "oiseau". Nous souhaitons créer une matrice d'incorporation pour ces mots en utilisant des vecteurs de dimension 2 (à des fins de simplification).

1. **Création du Vocabulaire :**
   - "chat"
   - "chien"
   - "oiseau"

2. **Définition des Vecteurs de Mots (GloVe) :**
   - "chat" : [0.8, 0.5]
   - "chien" : [0.7, 0.4]
   - "oiseau" : [0.2, 0.9]

3. **Création de la Matrice d'Incorporation :**
   - La matrice d'incorporation est une structure de données qui associe chaque mot à son vecteur de mots pré-entraînés. Dans cet exemple, nous utilisons une matrice 3x2.

4. **Initialisation de la Matrice avec des Zéros :**
   - Nous créons une matrice 3x2 initialisée avec des zéros.

5. **Remplissage de la Matrice d'Incorporation :**
   - Pour chaque mot de notre vocabulaire, nous essayons de récupérer le vecteur GloVe correspondant dans notre dictionnaire de vecteurs pré-entraînés. Si le vecteur GloVe existe, nous le copions dans la matrice d'incorporation sous l'index correspondant.

6. **Résultat de la Matrice d'Incorporation :**

   La matrice d'incorporation de sortie ressemble à ceci :
   
[[0.0, 0.0]

[0.8, 0.5]

[0.7, 0.4]

[0.2, 0.9]]


- La première ligne (index 0) est généralement réservée pour les mots inconnus ou les mots qui n'ont pas de vecteurs GloVe associés. Dans cet exemple, "chat" est à l'index 1, "chien" est à l'index 2 et "oiseau" est à l'index 3.

Cette matrice d'incorporation est ensuite utilisée pour initialiser les poids des couches d'incorporation dans un modèle de traitement de texte. Elle permet au modèle de comprendre la sémantique des mots en utilisant des vecteurs de mots pré-entraînés.


In [ ]:
embedding_layer = tf.keras.layers.Embedding(vocab_size,  # Taille du vocabulaire
                                          EMBEDDING_DIM,  # Dimension des vecteurs d'incorporation
                                          weights=[embedding_matrix],  # Matrice d'incorporation pré-entraînée
                                          input_length=MAX_SEQUENCE_LENGTH,  # Longueur maximale des séquences en entrée
                                          trainable=False)  # Les poids de cette couche ne sont pas entraînables


#### Rôle de la Couche d'Incorporation (Embedding Layer)

La couche d'incorporation (`embedding_layer`) joue un rôle essentiel dans le traitement de texte avec des réseaux de neurones. Son objectif principal est de convertir des mots (représentés par des entiers) en vecteurs de valeurs réelles, souvent appelés "vecteurs d'incorporation" ou "word embeddings". Voici son rôle en détail :

- **Transformation de Mots en Vecteurs :** La couche d'incorporation prend en entrée des mots sous forme d'entiers, généralement des indices de mots dans un vocabulaire. Elle attribue un vecteur d'incorporation unique à chaque mot de l'entrée. Ces vecteurs sont appris pendant l'entraînement du modèle et capturent la sémantique des mots.

- **Compréhension de la Sémantique :** Les vecteurs d'incorporation permettent au modèle de comprendre la signification des mots en les plaçant dans un espace vectoriel continu. Les mots similaires en termes de sens sont représentés par des vecteurs similaires, ce qui permet au modèle de généraliser à partir d'exemples d'entraînement à des mots qu'il n'a jamais vus.

- **Utilisation de Vecteurs Pré-entraînés :** Dans de nombreux cas, les vecteurs d'incorporation sont initialement pré-entraînés sur de grands corpus de texte, comme les vecteurs GloVe ou Word2Vec. Ces vecteurs pré-entraînés capturent la sémantique des mots à partir de vastes quantités de données textuelles et sont souvent utilisés comme points de départ pour les modèles de traitement de texte.

- **Fixation ou Entraînement des Vecteurs :** La couche d'incorporation peut être configurée pour fixer les vecteurs (non entraînables) ou les entraîner spécifiquement pour la tâche en cours. Les vecteurs pré-entraînés sont généralement figés, tandis que les vecteurs appris spécifiquement pour une tâche particulière sont ajustés pendant l'entraînement du modèle.

- **Première Couche d'un Modèle :** La couche d'incorporation est généralement la première couche d'un modèle de traitement de texte. Elle transforme les mots en vecteurs d'incorporation avant de les passer à d'autres couches du réseau de neurones, telles que les couches LSTM ou CNN pour la compréhension du texte.

En résumé, la couche d'incorporation joue un rôle crucial en permettant au modèle de traiter du texte sous une forme numérique et de comprendre la sémantique des mots. Elle est un élément clé pour améliorer les performances des modèles de traitement de texte dans de nombreuses applications, telles que la classification de texte, la génération de texte et la traduction automatique.


# Model Training - LSTM
We are clear to build our Deep Learning model. While developing a DL model, we should keep in mind of key things like Model Architecture, Hyperparmeter Tuning and Performance of the model.

As you can see in the word cloud, the some words are predominantly feature in both Positive and Negative tweets. This could be a problem if we are using a Machine Learning model like Naive Bayes, SVD, etc.. That's why we use **Sequence Models**.

### Sequence Model
![Sequence Model](https://miro.medium.com/max/1458/1*SICYykT7ybua1gVJDNlajw.png)

Reccurent Neural Networks can handle a seqence of data and learn a pattern of input seqence to give either sequence or scalar value as output. In our case, the Neural Network outputs a scalar value prediction. 

For model architecture, we use

1) **Embedding Layer** - Generates Embedding Vector for each input sequence.

2) **Conv1D Layer** - Its using to convolve data into smaller feature vectors. 

3) **LSTM** - Long Short Term Memory, its a variant of RNN which has memory state cell to learn the context of words which are at further along the text to carry contextual meaning rather than just neighbouring words as in case of RNN.

4) **Dense** - Fully Connected Layers for classification


In [ ]:
from tensorflow.keras.layers import Conv1D, Bidirectional, LSTM, Dense, Input, Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype='int32')
embedding_sequences = embedding_layer(sequence_input)
x = SpatialDropout1D(0.2)(embedding_sequences)
x = Conv1D(64, 5, activation='relu')(x)
x = Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2))(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(sequence_input, outputs)

### Optimization Algorithm
This notebook uses Adam, optimization algorithm for Gradient Descent. You can learn more about Adam [here](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)

### Callbacks
Callbacks are special functions which are called at the end of an epoch. We can use any functions to perform specific operation after each epoch. I used two callbacks here,

- **LRScheduler** - It changes a Learning Rate at specfic epoch to achieve more improved result. In this notebook, the learning rate exponentionally decreases after remaining same for first 10 Epoch.

- **ModelCheckPoint** - It saves best model while training based on some metrics. Here, it saves the model with minimum Validity Loss.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

model.compile(optimizer=Adam(learning_rate=LR), loss='binary_crossentropy',
              metrics=['accuracy'])
ReduceLROnPlateau = ReduceLROnPlateau(factor=0.1,
                                     min_lr = 0.01,
                                     monitor = 'val_loss',
                                     verbose = 1)

Let's start training... It takes a heck of a time if training in CPU, be sure your GPU turned on... May the CUDA Cores be with you....

In [ ]:
print("Training on GPU...") if tf.test.is_gpu_available() else print("Training on CPU...")

In [ ]:
history = model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS,
                    validation_data=(x_test, y_test), callbacks=[ReduceLROnPlateau])

# Model Evaluation
Now that we have trained the model, we can evaluate its performance. We will some evaluation metrics and techniques to test the model.

Let's start with the Learning Curve of loss and accuracy of the model on each epoch.

In [ ]:
s, (at, al) = plt.subplots(2,1)
at.plot(history.history['accuracy'], c= 'b')
at.plot(history.history['val_accuracy'], c='r')
at.set_title('model accuracy')
at.set_ylabel('accuracy')
at.set_xlabel('epoch')
at.legend(['LSTM_train', 'LSTM_val'], loc='upper left')

al.plot(history.history['loss'], c='m')
al.plot(history.history['val_loss'], c='c')
al.set_title('model loss')
al.set_ylabel('loss')
al.set_xlabel('epoch')
al.legend(['train', 'val'], loc = 'upper left')

The model will output a prediction score between 0 and 1. We can classify two classes by defining a threshold value for it. In our case, I have set 0.5 as THRESHOLD value, if the score above it. Then it will be classified as **POSITIVE** sentiment.

In [ ]:
def decode_sentiment(score):
    return "Positive" if score>0.5 else "Negative"


scores = model.predict(x_test, verbose=1, batch_size=10000)
y_pred_1d = [decode_sentiment(score) for score in scores]

### Confusion Matrix
Confusion Matrix provide a nice overlook at the model's performance in classification task

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
def plot_confusion_matrix(cm, classes,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """

    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=20)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, fontsize=13)
    plt.yticks(tick_marks, classes, fontsize=13)

    fmt = '.2f'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label', fontsize=17)
    plt.xlabel('Predicted label', fontsize=17)

In [ ]:
cnf_matrix = confusion_matrix(test_data.sentiment.to_list(), y_pred_1d)
plt.figure(figsize=(6,6))
plot_confusion_matrix(cnf_matrix, classes=test_data.sentiment.unique(), title="Confusion matrix")
plt.show()

### Classification Scores

In [ ]:
print(classification_report(list(test_data.sentiment), y_pred_1d))

It's a pretty good model we trained here in terms of NLP. Around 80% accuracy is good enough considering the baseline human accuracy also pretty low in these tasks. Also, you may go on and explore the dataset, some tweets might have other languages than English. So our Embedding and Tokenizing wont have effect on them. But on practical scenario, this model is good for handling most tasks for Sentiment Analysis.

<h3>Some of the resource and people who help me learn some concepts</h3>
<font color='#008080'>
    <ul>
        <li> <b>Andrew NG's Seqence Model Course</b> at <a href="https://www.coursera.org/learn/nlp-sequence-models"> Coursera</a> </li>
    
<li> <b>Andrej Karpathy's Blog</b> on <a href="http://karpathy.github.io/2015/05/21/rnn-effectiveness/">Effectiveness of RNN</a></li>

<li> <b>Intuitive Understanding of GloVe Embedding</b> on <a href="https://towardsdatascience.com/light-on-math-ml-intuitive-guide-to-understanding-glove-embeddings-b13b4f19c010">TDS</a></li>

<li> <b>Keras tutorial on Word Embedding</b> <a href="https://blog.keras.io/using-pre-trained-word-embeddings-in-a-keras-model.html"> here</a></li>

</ul>
</font>

> <font color='#696969'>I got to say like you, I am still at learning phase in terms of NLP. I have got lot to learn in future. I found that writing this notebook even though it is done by lot of people before me helps me with a deeper and complete understanding our the concepts that I am learning. Kaggle has been a amazing place to learn from and contribute to community of Data Science Aspirants.</font>

<h2><font color='red'> If you find this notebook usefull kindly UPVOTE this notebook. I am new to writting notebooks hope that would really encourage me to write and learn more.</font></h2>

<h5>Thanks in Advance. Have a nice day. Learn more and Happy Kaggle</h5>